# Data SAIL Club Notebook #2
# First Contact with Scikit-learn
This notebook picks up right where the pandas one left off.
You already have a `df` in memory. Now you will build your first machine learning models with it.


---
## 0. The Idea in One Paragraph

Scikit-learn follows a single pattern for every model:

1. **Prepare** your data — a table of numbers, no NaN
2. **Split** into train / test sets
3. **Choose** a model and call `.fit(X_train, y_train)`
4. **Predict** with `.predict(X_test)`
5. **Evaluate** with a metric

That loop is the same whether you use a decision tree or a neural network.

---
## 1. Re-connect to Your Data

Load your dataframe again here (the new one you saved after finishing the notebook)

In [ ]:
import pandas as pd
import numpy as np

# FILE = "output.csv"
# df = pd.read_csv(FILE)

print(df.shape)
df.head(3)

---
## 2. Pick a Target

Every supervised model needs:
- **X** — the features (input columns)
- **y** — the target (the column you want to predict)

Two common tasks:
- **Regression** — target is a continuous number (price, temperature, score)
- **Classification** — target is a category (yes/no, species, label)

In [ ]:
print(df.columns)
TARGET = "FG" # CHANGE to column you want to predict

# FEATURES = [c for c in df.columns if c != TARGET] # all except the target

FEATURES = ['MP','AST','TOV'] # OR choose only a few

print("Target  :", TARGET)
print("Features:", FEATURES)

Index(['Player', 'Tm', 'Opp', 'Res', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA',
       '3P%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK',
       'TOV', 'PF', 'PTS', 'GmSc', 'Data'],
      dtype='object')
Target  : FG
Features: ['MP', 'AST', 'TOV']


---
## 3. Prepare the Data

Scikit-learn requires:
- **No missing values**
- **Numbers only** (no raw strings)

We handle both below with the minimum amount of code.

In [ ]:
# Step 1: keep only the columns we care about
data = df[FEATURES + [TARGET]].copy()

# Step 2: drop rows with any missing value (simplest approach)
data = data.dropna()
print(f"{len(data)} rows after dropping NaN (was {len(df)})")

In [ ]:
# Step 3: encode text columns as numbers
# pd.get_dummies turns each category into a 0/1 column (one-hot encoding)
data = pd.get_dummies(data)

print("Columns after encoding:", list(data.columns))

In [ ]:
# Step 4: split into X (features) and y (target)
# After get_dummies the target column name may have changed if it was a string;
# find it by checking which columns start with TARGET
target_cols = [c for c in data.columns if c == TARGET or c.startswith(TARGET + "_")]

X = data.drop(columns=target_cols)
y = data[TARGET] if TARGET in data.columns else data[target_cols[0]]

print("X shape:", X.shape)
print("y shape:", y.shape)

---
## 4. Train / Test Split

We hold out 20% of the data as a test set.
The model never sees it during training — that is how we measure real performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% goes to test
    random_state=42   # fix the seed for reproducibility
)

print("Train size:", len(X_train))
print("Test  size:", len(X_test))

---
## 5. First Model: Decision Tree.


A decision tree is a great first model — it makes no assumptions, requires no scaling,
and its predictions are easy to reason about.

We will automatically detect whether the task is regression or classification.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# Decide task type based on the number of unique target values
n_unique = y.nunique()
task = "classification" if n_unique <= 20 else "regression" # If less than 20 unique values we will make it classifiaation
print(f"Detected task: {task}  (target has {n_unique} unique values)")

# Pick the right model
if task == "classification":
    model = DecisionTreeClassifier(max_depth=4, random_state=42)
else:
    model = DecisionTreeRegressor(max_depth=4, random_state=42)

In [ ]:
# Train the model -- this is always just one line
model.fit(X_train, y_train)

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

# Show the first 10 predictions vs. actual values
comparison = pd.DataFrame({"actual": y_test.values[:10], "predicted": y_pred[:10]})
comparison

---
## 6. Evaluate the Model

A single number to say how good the model is.

In [ ]:
if task == "classification":
    from sklearn.metrics import accuracy_score, classification_report

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.2%}")
    print()
    # Precision, recall, F1 per class
    print(classification_report(y_test, y_pred))

else:
    from sklearn.metrics import mean_absolute_error, r2_score

    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    print(f"Mean Absolute Error : {mae:.4f}")
    print(f"R2 score            : {r2:.4f}  (1.0 = perfect, 0 = baseline)")

---
## 7. Feature Importance

Which columns mattered most for the predictions?

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().tail(15).plot(kind="barh")
plt.title("Feature Importances")
plt.tight_layout()
plt.show()

---
## 8. Trying a Second Model

Because scikit-learn uses the same API for every model,
switching models is just swapping the class name.
Everything else (split, fit, predict, evaluate) stays identical.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Random Forest = many decision trees averaged together, usually more accurate
if task == "classification":
    model2 = RandomForestClassifier(n_estimators=100, random_state=42)
else:
    model2 = RandomForestRegressor(n_estimators=100, random_state=42)

model2.fit(X_train, y_train)
y_pred2 = model2.predict(X_test)

if task == "classification":
    print("Random Forest accuracy:", f"{accuracy_score(y_test, y_pred2):.2%}")
else:
    print("Random Forest MAE:", f"{mean_absolute_error(y_test, y_pred2):.4f}")
    print("Random Forest R2: ", f"{r2_score(y_test, y_pred2):.4f}")

---
## 9. Feature Scaling (when it matters)

Tree-based models (decision tree, random forest) do not need scaling.
Distance-based models (KNN, SVM, linear regression with regularisation) do.

Standard scaling: subtract mean, divide by standard deviation — each feature ends up around 0.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# Scale ONLY on the training data, then apply the same transform to test
# (fitting on test data would be data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

if task == "classification":
    model3 = KNeighborsClassifier(n_neighbors=5)
else:
    model3 = KNeighborsRegressor(n_neighbors=5)

model3.fit(X_train_scaled, y_train)
y_pred3 = model3.predict(X_test_scaled)

if task == "classification":
    print("KNN accuracy:", f"{accuracy_score(y_test, y_pred3):.2%}")
else:
    print("KNN MAE:", f"{mean_absolute_error(y_test, y_pred3):.4f}")
    print("KNN R2: ", f"{r2_score(y_test, y_pred3):.4f}")

---
## 10. Cross-Validation

A single train/test split can be lucky or unlucky depending on which rows ended up where.
Cross-validation repeats the split k times and averages the scores — a much more reliable estimate.

In [ ]:
from sklearn.model_selection import cross_val_score

# 5-fold CV on the Random Forest
scoring = "accuracy" if task == "classification" else "r2"

scores = cross_val_score(model2, X, y, cv=5, scoring=scoring)

print(f"CV scores ({scoring}): {scores.round(3)}")
print(f"Mean: {scores.mean():.3f}  |  Std: {scores.std():.3f}")

---
## 11. Saving and Loading a Model

So you do not have to retrain every time.

In [ ]:
import joblib

# Save
# joblib.dump(model2, "my_model.pkl")

# Load later
# model2 = joblib.load("my_model.pkl")

print("Uncomment to save/load the model.")

---
## Cheat Sheet

| Step | Code |
|---|---|
| Drop NaN | `df.dropna()` |
| Encode categories | `pd.get_dummies(df)` |
| Split | `train_test_split(X, y, test_size=0.2)` |
| Train | `model.fit(X_train, y_train)` |
| Predict | `model.predict(X_test)` |
| Accuracy | `accuracy_score(y_test, y_pred)` |
| Regression error | `mean_absolute_error(y_test, y_pred)` |
| Scale features | `StandardScaler().fit_transform(X_train)` |
| Cross-validate | `cross_val_score(model, X, y, cv=5)` |
| Save model | `joblib.dump(model, "model.pkl")` |

---
## Where to Go Next

- **Pipelines** — chain preprocessing + model into one object (`sklearn.pipeline.Pipeline`)
- **Hyperparameter tuning** — `GridSearchCV` or `RandomizedSearchCV` to find the best settings
- **More models** — `LogisticRegression`, `SVC`, `GradientBoostingClassifier`, `LinearRegression`
- **Unsupervised** — `KMeans` for clustering, `PCA` for dimensionality reduction
- **Docs** — https://scikit-learn.org/stable/user_guide.html